# SHAP & Counterfactual Explainability — NBA Shooting Biomechanics

**Branch:** `phillip/shap_model`

This notebook completes the interpretability layer of the capstone's causal chain:

```
Biomechanics  →  Ball Flight  →  Miss Distance
   (this notebook: SHAP + Counterfactuals on both links)
```

**Primary model:** Sophie's LightGBM (biomechanics → ball flight, R²=0.70–0.95)  
**Secondary model:** Random Forest (ball flight → miss distance, completes the chain)

**Sections:**
1. Data Preprocessing (reproduce Sophie's handedness normalization)
2. Model Training (6 LightGBM models, one per ball flight target)
3. Global SHAP Analysis (which biomechanical features drive each ball flight parameter?)
4. Local SHAP Analysis (per-player waterfall explanations)
5. Counterfactual Explanations (DiCE: what biomechanical changes would improve a player?)
6. SHAP on RF → Miss Distance (complete the causal chain)
7. Synthesis & Coaching Insights

## Section 0: Setup

In [ ]:
!pip install shap lightgbm dice-ml -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, re, joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import shap
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import dice_ml

shap.initjs()
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')
print('All imports successful.')

In [ ]:
# Data is in project root, one level up from this notebook
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'capstone2026v2.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = '../capstone2026v2.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {df_raw.shape[0]:,} shots x {df_raw.shape[1]} columns')
print(f'Players: {df_raw["PlayerID"].nunique()}')
df_raw.head(2)

## Section 1: Data Preprocessing

Reproduces Sophie's pipeline from `Ball_Flight_to_Biomechanics.ipynb`:
- **Handedness normalization:** left-handed players are reflected into a right-handed reference frame so all 'Dominant side' features are consistent
- **Feature selection:** keyword-based filter selects biomechanical variables (ankle, knee, hip, torso, etc.) and excludes non-biomechanical ones (ball position, time, shot metadata)

In [ ]:
def _reflect_across_shot_axis(x, y):
    norm = np.sqrt(x**2 + y**2)
    if norm == 0:
        return x, y
    ux, uy = x / norm, y / norm
    proj = x * ux + y * uy
    px, py = proj * ux, proj * uy
    return x - 2 * (x - px), y - 2 * (y - py)


def standardize_handedness(df):
    """Normalize left-handed players to right-handed reference frame."""
    df = df.copy()
    left_mask = df['hand'] == 'Left'

    # Reflect shot location
    coords = df.loc[left_mask, ['Shot.X.Pos', 'Shot.Y.Pos']].values
    if len(coords) > 0:
        reflected = np.array([_reflect_across_shot_axis(x, y) for x, y in coords])
        df.loc[left_mask, 'Shot.X.Pos'] = reflected[:, 0]
        df.loc[left_mask, 'Shot.Y.Pos'] = reflected[:, 1]
    df.loc[left_mask, 'Shot.X.Pos'] *= -1

    # Apply to all X/Y coordinate column pairs
    x_cols = [c for c in df.columns if re.search(r'(^|_)X($|[A-Z])', c)]
    y_cols = [c for c in df.columns if re.search(r'(^|_)Y($|[A-Z])', c)]
    for x_col, y_col in zip(x_cols, y_cols):
        coords = df.loc[left_mask, [x_col, y_col]].values
        if len(coords) > 0:
            reflected = np.array([_reflect_across_shot_axis(x, y) for x, y in coords])
            df.loc[left_mask, x_col] = reflected[:, 0]
            df.loc[left_mask, y_col] = reflected[:, 1]
        df.loc[left_mask, x_col] *= -1

    # Swap Left/Right biomechanical columns for left-handed players
    left_cols = [c for c in df.columns if 'Left' in c]
    for left_col in left_cols:
        right_col = left_col.replace('Left', 'Right')
        if right_col in df.columns:
            temp = df.loc[left_mask, left_col].copy()
            df.loc[left_mask, left_col] = df.loc[left_mask, right_col]
            df.loc[left_mask, right_col] = temp

    # Rename Left/Right -> NonDominant/Dominant
    rename_dict = {}
    for c in df.columns:
        if 'Right' in c:
            rename_dict[c] = c.replace('Right', 'Dominant')
        elif 'Left' in c:
            rename_dict[c] = c.replace('Left', 'NonDominant')
    df = df.rename(columns=rename_dict)
    df['hand_standardized'] = 'Dominant'
    return df


BIO_KEYWORDS = [
    'ankle', 'knee', 'hip', 'shoulder', 'elbow', 'torso',
    'flexion', 'extension', 'alignment', 'rom', 'velocity', 'velo',
    'dorsiflexion', 'plantarflexion', 'stance', 'gap'
]
EXCLUDE_KEYWORDS = ['ball', 'time', 'shot', 'made', 'x', 'y', 'z', 'pos', 'distance']


def select_biomechanical_features(df):
    selected = []
    for c in df.columns:
        c_low = c.lower()
        if any(ex in c_low for ex in EXCLUDE_KEYWORDS):
            continue
        if any(bio in c_low for bio in BIO_KEYWORDS):
            selected.append(c)
    return selected

In [ ]:
df_std1 = standardize_handedness(df_raw)

biomech_features = select_biomechanical_features(df_std1)
X_full = df_std1[biomech_features].select_dtypes(include=[np.number])

print(f'Biomechanical features selected: {len(X_full.columns)}')
print(f'Sample features: {list(X_full.columns[:10])}')

## Section 2: Model Training — LightGBM (Biomechanics → Ball Flight)

Six separate `LGBMRegressor` models, one per ball flight target. Models are saved to `saved_models/` so they only need to be trained once — subsequent runs load from disk.

Expected R² (from Sophie's notebook): Ball_to_Rim_Angle≈0.70, Ball_Velocity≈0.84, Release.Height≈0.87, Initial.Ball.Angle≈0.79, **Initial.Ball.Velocity≈0.95**, Max.Ball.Arc≈0.88

In [ ]:
BALL_FLIGHT_TARGETS = [
    'Ball_to_Rim_Angle',
    'Ball_Velocity',
    'Release.Height',
    'Initial.Ball.Angle',
    'Initial.Ball.Velocity',
    'Max.Ball.Arc'
]

MODEL_DIR = 'saved_models'
os.makedirs(MODEL_DIR, exist_ok=True)

models        = {}
metrics       = {}
X_per_target  = {}
y_per_target  = {}

for target in BALL_FLIGHT_TARGETS:
    model_path = os.path.join(MODEL_DIR, f'lgbm_{target.replace(".", "_")}.pkl')

    data    = pd.concat([X_full, df_std1[target]], axis=1).dropna()
    X_clean = data[X_full.columns]
    y_clean = data[target]

    if os.path.exists(model_path):
        model = joblib.load(model_path)
        source = 'loaded'
    else:
        model = LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1)
        model.fit(X_clean, y_clean)
        joblib.dump(model, model_path)
        source = 'trained'

    preds = model.predict(X_clean)
    r2    = r2_score(y_clean, preds)
    mae   = mean_absolute_error(y_clean, preds)
    rmse  = np.sqrt(mean_squared_error(y_clean, preds))

    models[target]       = model
    metrics[target]      = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    X_per_target[target] = X_clean
    y_per_target[target] = y_clean

    print(f'[{source}] {target:28s}  R²={r2:.3f}  MAE={mae:.4f}  RMSE={rmse:.4f}')

In [ ]:
metrics_df = pd.DataFrame(metrics).T.round(4)
metrics_df.index.name = 'Target'
metrics_df.style.background_gradient(subset=['R2'], cmap='RdYlGn').format('{:.4f}')

## Section 3: Global SHAP Analysis

`shap.TreeExplainer` computes exact Shapley values for tree-based models — no approximation. For each of the 6 LightGBM models we produce:
- **Beeswarm plot:** each dot is one shot; x-axis = SHAP value (impact on prediction); color = feature value (red=high, blue=low)
- **Cross-model summary:** which biomechanical features consistently appear across multiple ball flight targets?

Computations are capped at 5,000 shots for speed (full dataset used if smaller).

In [ ]:
SHAP_SAMPLE = 5000

shap_values_dict  = {}
explainers_dict   = {}
shap_samples_dict = {}

for target in BALL_FLIGHT_TARGETS:
    X_clean  = X_per_target[target]
    X_sample = X_clean.sample(min(SHAP_SAMPLE, len(X_clean)), random_state=42)

    explainer = shap.TreeExplainer(models[target])
    shap_vals = explainer.shap_values(X_sample)

    shap_values_dict[target]  = shap_vals
    explainers_dict[target]   = explainer
    shap_samples_dict[target] = X_sample

    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_vals, X_sample, max_display=15, show=False)
    plt.title(f'Global SHAP Beeswarm — {target}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# Build cross-model mean |SHAP| table
rank_series = []
for target in BALL_FLIGHT_TARGETS:
    sv       = shap_values_dict[target]
    X_sample = shap_samples_dict[target]
    mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=X_sample.columns, name=target)
    rank_series.append(mean_abs)

cross_df = pd.concat(rank_series, axis=1)
cross_df['mean_rank'] = cross_df.mean(axis=1)
cross_df = cross_df.sort_values('mean_rank', ascending=False)

print('Top 20 biomechanical features by mean |SHAP| across all 6 ball flight targets:')
display(
    cross_df.head(20)
    .drop(columns='mean_rank')
    .style.background_gradient(cmap='YlOrRd')
    .format('{:.4f}')
)

In [ ]:
top15 = cross_df.head(15).index

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(
    cross_df.loc[top15, BALL_FLIGHT_TARGETS],
    cmap='YlOrRd', ax=ax, linewidths=0.4,
    annot=True, fmt='.3f', annot_kws={'size': 7}
)
ax.set_title(
    'Mean |SHAP| — Top 15 Biomechanical Features x Ball Flight Targets\n'
    'Higher = stronger causal contribution to that ball flight parameter',
    fontsize=12
)
ax.set_xlabel('Ball Flight Target')
ax.set_ylabel('Biomechanical Feature')
plt.tight_layout()
plt.show()

## Section 4: Local SHAP Analysis — Player Level

**Waterfall plots** decompose a single shot (or player average) into additive SHAP contributions:
- Base value = model's expected prediction across all training shots
- Each bar shows how much one feature pushed the prediction up (red) or down (blue)
- Final value = base + sum of all SHAP values = model's prediction

We compare the best and worst shooters using `Initial.Ball.Velocity` (highest R²=0.95).

In [ ]:
focus_target = 'Initial.Ball.Velocity'
X_clean      = X_per_target[focus_target]
explainer    = explainers_dict[focus_target]

# Join back to player info using the preserved index
player_made = df_std1.loc[X_clean.index, ['PlayerID', 'Made']].copy()

# Filter to players with >= 20 shots for stable averages
player_counts    = player_made.groupby('PlayerID').size()
qualified        = player_counts[player_counts >= 20].index
player_made_q    = player_made[player_made['PlayerID'].isin(qualified)]
player_make_rate = player_made_q.groupby('PlayerID')['Made'].mean()

top_player    = player_make_rate.idxmax()
bottom_player = player_make_rate.idxmin()

print(f'Best shooter:  PlayerID={top_player},  make rate={player_make_rate[top_player]:.3f}')
print(f'Worst shooter: PlayerID={bottom_player}, make rate={player_make_rate[bottom_player]:.3f}')

In [ ]:
player_ids = df_std1.loc[X_clean.index, 'PlayerID']

for pid, label in [(top_player, 'Best Shooter'), (bottom_player, 'Worst Shooter')]:
    mask     = player_ids == pid
    player_X = X_clean[mask]

    if len(player_X) == 0:
        print(f'No data for {label} (PlayerID={pid})')
        continue

    rep_shot  = player_X.median()  # representative shot = player's median biomechanics
    sv_single = explainer.shap_values(rep_shot.values.reshape(1, -1))

    explanation = shap.Explanation(
        values=sv_single[0],
        base_values=explainer.expected_value,
        data=rep_shot.values,
        feature_names=list(X_clean.columns)
    )

    pred_val  = models[focus_target].predict(rep_shot.values.reshape(1, -1))[0]
    make_rate = player_make_rate.get(pid, float('nan'))

    plt.figure()
    shap.waterfall_plot(explanation, max_display=12, show=False)
    plt.title(
        f'SHAP Waterfall — {label} (PlayerID: {pid})\n'
        f'Target: {focus_target} | Predicted: {pred_val:.3f} m/s | Make Rate: {make_rate:.3f}',
        fontsize=11
    )
    plt.tight_layout()
    plt.show()
    print()

## Section 5: Counterfactual Explanations (DiCE)

**Counterfactual explanations** answer: *"What minimal changes to this player's biomechanics would push their predicted ball flight into the optimal range?"*

We use [DiCE (Diverse Counterfactual Explanations)](https://github.com/interpretml/DiCE) with `method='random'`. Key design choices:
- Target: `Initial.Ball.Velocity` (highest R²=0.95 — most reliable model)
- Desired range: top 25th percentile of Initial Ball Velocity values
- Features varied: top 10 biomechanical features from SHAP (more tractable, more interpretable)
- Query player: a player whose predicted ball velocity is currently below the dataset median

In [ ]:
focus_target    = 'Initial.Ball.Velocity'
X_clean         = X_per_target[focus_target]
y_clean         = y_per_target[focus_target]
model           = models[focus_target]

# Restrict counterfactual search to top 10 SHAP features for interpretability
top_shap_feats = [f for f in cross_df.head(10).index if f in X_clean.columns]
print(f'Features allowed to vary in counterfactuals ({len(top_shap_feats)}):')
for f in top_shap_feats:
    print(f'  {f}')

# Sample 2000 rows for DiCE data interface
dice_sample = pd.concat([X_clean, y_clean], axis=1).dropna().sample(
    min(2000, len(X_clean)), random_state=42
).reset_index(drop=True)

d = dice_ml.Data(
    dataframe=dice_sample,
    continuous_features=list(X_clean.columns),
    outcome_name=focus_target
)
m       = dice_ml.Model(model=model, backend='sklearn')
exp_cf  = dice_ml.Dice(d, m, method='random')

target_min = float(y_clean.quantile(0.75))
target_max = float(y_clean.max())
print(f'\nTarget range (top 25%): [{target_min:.3f}, {target_max:.3f}] m/s')
print(f'Dataset median: {float(y_clean.median()):.3f} m/s')

In [ ]:
# Find a player whose predicted ball velocity is currently below the median
preds_all  = model.predict(X_clean)
pred_series = pd.Series(preds_all, index=X_clean.index)
player_ids = df_std1.loc[X_clean.index, 'PlayerID']

below_median_mask = pred_series < float(y_clean.median())
improvement_player = (
    player_ids[below_median_mask]
    .value_counts()
    .idxmax()
)

player_shots = X_clean[player_ids == improvement_player]
query_shot   = player_shots.median().to_frame().T

current_pred = model.predict(query_shot)[0]
print(f'Query player: PlayerID={improvement_player}')
print(f'Current predicted {focus_target}: {current_pred:.3f} m/s')
print(f'Goal: reach top 25% range [{target_min:.3f}, {target_max:.3f}] m/s')
print(f'Required improvement: +{target_min - current_pred:.3f} m/s')

In [ ]:
cf_result = exp_cf.generate_counterfactuals(
    query_instances=query_shot,
    total_CFs=5,
    desired_range=[target_min, target_max],
    features_to_vary=top_shap_feats,
    random_seed=42
)

print(f'Counterfactual scenarios for PlayerID={improvement_player}')
print(f'Each row shows what biomechanical values this player would need to reach optimal ball velocity.\n')
cf_result.visualize_as_dataframe(show_only_changes=True)

## Section 6: SHAP on RF → Miss Distance (Complete the Causal Chain)

Now we apply SHAP to the second link: **ball flight → miss distance**.

One Random Forest is trained per shot location × type combination (same setup as Sophie's notebook). SHAP values reveal *which ball flight parameters matter most for accuracy at each shot context* — completing the interpretability chain:

```
Biomechanics  →[SHAP]→  Ball Flight  →[SHAP]→  Miss Distance
```

In [ ]:
PREDICTORS = [
    'Ball_to_Rim_Angle',
    'Ball_Velocity',
    'Release.Height.Norm',
    'Initial.Ball.Angle',
    'Initial.Ball.Velocity',
    'Max.Ball.Arc'
]
BDFC_TARGET = 'Ball.Distance.from.Center'
MIN_N       = 100

# Compute normalized release height if not already present
if 'Release.Height.Norm' not in df_std1.columns:
    height_col = 'height' if 'height' in df_std1.columns else 'Height'
    df_std1['Release.Height.Norm'] = df_std1['Release.Height'] / df_std1[height_col]

# Build location x type groups
groups = []
for loc in df_std1['Shot.Location'].unique():
    if loc == 'Free Throw':
        subset = df_std1[df_std1['Shot.Location'] == loc].copy()
        if len(subset) >= MIN_N:
            groups.append((loc, 'All', subset))
    else:
        for shot_type in df_std1[df_std1['Shot.Location'] == loc]['Shot.Type'].unique():
            subset = df_std1[
                (df_std1['Shot.Location'] == loc) &
                (df_std1['Shot.Type'] == shot_type)
            ].copy()
            if len(subset) >= MIN_N:
                groups.append((loc, shot_type, subset))

print(f'Shot groups to model: {len(groups)}')

rf_shap_rows = []

for loc, shot_type, subset in groups:
    data = subset[PREDICTORS + [BDFC_TARGET]].dropna()
    if len(data) < MIN_N:
        continue

    rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(data[PREDICTORS], data[BDFC_TARGET])
    r2 = r2_score(data[BDFC_TARGET], rf.predict(data[PREDICTORS]))

    exp_rf  = shap.TreeExplainer(rf)
    sv_rf   = exp_rf.shap_values(data[PREDICTORS])
    mean_abs = pd.Series(np.abs(sv_rf).mean(axis=0), index=PREDICTORS)

    row = mean_abs.to_dict()
    row.update({'location': loc, 'shot_type': shot_type, 'r2': round(r2, 3), 'n': len(data)})
    rf_shap_rows.append(row)

    print(f'{loc:20s} | {shot_type:20s}  R²={r2:.3f}  n={len(data)}')

rf_shap_df = pd.DataFrame(rf_shap_rows)

In [ ]:
rf_shap_df['group'] = rf_shap_df['location'] + ' | ' + rf_shap_df['shot_type']
pivot = rf_shap_df.set_index('group')[PREDICTORS]

fig, ax = plt.subplots(figsize=(12, max(6, len(pivot) * 0.5)))
sns.heatmap(
    pivot, cmap='Blues', ax=ax, linewidths=0.4,
    annot=True, fmt='.3f', annot_kws={'size': 7}
)
ax.set_title(
    'Mean |SHAP| — Ball Flight Features → Miss Distance, by Shot Context\n'
    'Higher = this ball flight parameter drives accuracy more at this location/type',
    fontsize=12
)
ax.set_xlabel('Ball Flight Feature')
ax.set_ylabel('Shot Context')
plt.tight_layout()
plt.show()

## Section 7: Synthesis & Coaching Insights

Pulling together SHAP from both model layers into actionable coaching recommendations.

In [ ]:
# Full causal chain heatmap: top 15 biomechanical features x 6 ball flight targets
top15 = cross_df.head(15).index

fig, ax = plt.subplots(figsize=(14, 8))
heatmap_data = cross_df.loc[top15, BALL_FLIGHT_TARGETS]
sns.heatmap(
    heatmap_data, cmap='YlOrRd', ax=ax, linewidths=0.4,
    annot=True, fmt='.3f', annot_kws={'size': 8}
)
ax.set_title(
    'SHAP Summary — Biomechanical Features Driving Ball Flight\n'
    'Read left-to-right: ankle ROM drives Initial Ball Velocity most, etc.',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Ball Flight Target', fontsize=11)
ax.set_ylabel('Biomechanical Feature', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Per-target: top biomechanical driver + direction + model quality
summary_rows = []
for target in BALL_FLIGHT_TARGETS:
    sv       = shap_values_dict[target]
    X_sample = shap_samples_dict[target]
    mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=X_sample.columns)
    mean_dir = pd.Series(sv.mean(axis=0), index=X_sample.columns)

    top_feat  = mean_abs.idxmax()
    top_val   = mean_abs.max()
    direction = '+' if mean_dir[top_feat] > 0 else '-'
    r2        = metrics[target]['R2']

    summary_rows.append({
        'Ball Flight Target'       : target,
        'Top Biomechanical Driver' : top_feat,
        'Direction'                : direction,
        'Mean |SHAP|'              : round(top_val, 4),
        'Model R²'                 : round(r2, 3)
    })

summary_df = pd.DataFrame(summary_rows)
print('Coaching Summary: Top biomechanical driver per ball flight target')
print('Direction: + means higher feature value → higher target value')
display(summary_df.style.hide(axis='index').background_gradient(subset=['Model R²'], cmap='RdYlGn'))

In [ ]:
# Average RF SHAP across all groups: which ball flight feature reduces miss distance most?
avg_rf_shap = rf_shap_df[PREDICTORS].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
avg_rf_shap.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(
    'Average |SHAP| — Ball Flight Features → Miss Distance\n'
    '(averaged across all shot location x type groups)',
    fontsize=11
)
ax.set_xlabel('Mean |SHAP| contribution to miss distance prediction')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nFull Causal Chain Interpretation:')
print('1. The biomechanical features above (Section 3) drive ball flight parameters')
print('2. Ball flight parameters above drive miss distance')
print('3. Counterfactuals (Section 5) prescribe the minimal biomechanical changes needed')
print('\nExample chain: ankle ROM -> Initial Ball Velocity -> reduced miss distance')